# Cancer Survival Prediction — TCGA-BRCA (Python: lifelines)

Kaplan–Meier + log-rank + **Cox proportional-hazards** + risk stratification on **real TCGA-BRCA** clinical data (Pan-Cancer Atlas 2018, from cBioPortal). Companion R version (`cancer_survival.R`, survival + survminer) uses the same cohort and agrees.

In [ ]:
%pip install lifelines -q
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
os.makedirs("results_py", exist_ok=True)

url = "https://media.githubusercontent.com/media/cBioPortal/datahub/master/public/brca_tcga_pan_can_atlas_2018/data_clinical_patient.txt"
cl = pd.read_csv(url, sep="\t", comment="#")
cl["time"]  = pd.to_numeric(cl["OS_MONTHS"], errors="coerce")
cl["event"] = cl["OS_STATUS"].astype(str).str.contains("1|DECEASED").astype(int)
cl["age"]   = pd.to_numeric(cl["AGE"], errors="coerce")
st = cl["AJCC_PATHOLOGIC_TUMOR_STAGE"].astype(str).str.upper()
cl["stage"] = np.select([st.str.contains("IV"), st.str.contains("III"), st.str.contains("II"), st.str.contains("I")],
                         [4, 3, 2, 1], default=np.nan)
df = cl.dropna(subset=["time", "event", "age", "stage", "SUBTYPE"]).copy()
df = df[df["time"] > 0]
df["stage_group"] = np.where(df["stage"] >= 3, "late (III-IV)", "early (I-II)")
print("patients:", len(df), "| deaths:", int(df['event'].sum()), "| censored:", int((df['event']==0).sum()))
print(df[["time", "event", "age", "stage", "SUBTYPE"]].head().to_string())

## 1. Kaplan–Meier by stage group + log-rank

In [ ]:
kmf = KaplanMeierFitter(); ax = plt.subplot(111)
for lab, g in df.groupby("stage_group"):
    kmf.fit(g["time"], event_observed=g["event"], label=lab); kmf.plot_survival_function(ax=ax)
plt.xlabel("months"); plt.ylabel("S(t)"); plt.title("TCGA-BRCA overall survival by stage")
plt.savefig("results_py/km_stage.png", dpi=150, bbox_inches="tight"); plt.show()
e, l = df[df.stage_group.str.startswith("early")], df[df.stage_group.str.startswith("late")]
lr = logrank_test(e["time"], l["time"], event_observed_A=e["event"], event_observed_B=l["event"])
print("log-rank p =", lr.p_value)

## 2. Multivariable Cox proportional-hazards (hazard ratios)

In [ ]:
model = pd.get_dummies(df[["time", "event", "age", "stage", "SUBTYPE"]], columns=["SUBTYPE"], drop_first=True)
cph = CoxPHFitter()
cph.fit(model, duration_col="time", event_col="event")
cph.print_summary()
print(cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].to_string())
cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].to_csv("results_py/cox_hr.csv")

## 3. C-index + risk stratification

In [ ]:
print("C-index:", round(cph.concordance_index_, 3))
model["risk"]  = cph.predict_partial_hazard(model)
model["group"] = np.where(model["risk"] > model["risk"].median(), "high risk", "low risk")
kmf = KaplanMeierFitter(); ax = plt.subplot(111)
for lab, g in model.groupby("group"):
    kmf.fit(g["time"], g["event"], label=lab); kmf.plot_survival_function(ax=ax)
plt.xlabel("months"); plt.ylabel("S(t)"); plt.title("KM by Cox risk group")
plt.savefig("results_py/km_riskgroups.png", dpi=150, bbox_inches="tight"); plt.show()

## Interpretation

Late-stage (III–IV) TCGA-BRCA patients have markedly worse survival than early-stage (significant log-rank). In the Cox model **tumour stage** carries the largest hazard ratio (HR > 1), **age** adds a modest HR > 1, and molecular **subtype** modulates prognosis (Luminal A best, Basal/HER2 worse). The **C-index (~0.65–0.75)** shows the model ranks risk usefully, and the median-split risk groups separate on KM. This is the basis of clinical staging + precision oncology. R (`survival`) and Python (`lifelines`) agree on the same real cohort.

**Caveats:** observational (association, not causal); proportional-hazards may not hold for every covariate; single cohort; C-index measures ranking, not calibrated risk.

## Abstract

*Built a breast-cancer overall-survival model on real TCGA-BRCA clinical data (Pan-Cancer Atlas 2018) in both R (survival) and Python (lifelines). Kaplan–Meier + log-rank showed a strong stage effect; a multivariable Cox model quantified stage, age, and PAM50 subtype as prognostic factors (C-index ~0.65–0.75), checked the proportional-hazards assumption, and stratified patients into separated risk groups — a compact precision-oncology prognostic pipeline, cross-validated across two toolchains.*